# PINN Review 2019–2026 — Shared Setup

Common imports, network architectures, training helpers and animation utilities used across all paper reproductions in this series.

This setup cell must be run before any other notebook in the PINN Review series.


## Setup — Shared Imports and Utilities


In [ ]:
import math, time, random, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
torch.set_default_dtype(torch.float32)
SEED=7
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cpu'

def rel_l2(yhat,y):
    return float(torch.linalg.norm(yhat-y)/torch.linalg.norm(y))

class MLP(nn.Module):
    def __init__(self, din=1, dout=1, width=32, depth=3, dropout=0.0, act=nn.Tanh):
        super().__init__(); layers=[]; d=din
        for _ in range(depth):
            layers += [nn.Linear(d,width), act()]
            if dropout: layers += [nn.Dropout(dropout)]
            d=width
        layers += [nn.Linear(d,dout)]
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def grad(y,x,order=1):
    g=y
    for _ in range(order):
        g=torch.autograd.grad(g,x,torch.ones_like(g),create_graph=True,retain_graph=True)[0]
    return g

def train(loss_fn, model, steps=600, lr=2e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); hist=[]
    t=time.time()
    for i in range(steps):
        opt.zero_grad(); loss=loss_fn(); loss.backward(); opt.step()
        if i % max(1,steps//100)==0: hist.append(float(loss.detach()))
    return hist, time.time()-t

results=[]
def record(paper,method,error,seconds,notes):
    results.append(dict(paper=paper,method=method,relative_L2=error,seconds=seconds,notes=notes))
